# Análise Exploratória de Dados

- Conjunto de Dados [`Sleep Efficiency Dataset`](https://www.kaggle.com/datasets/equilibriumm/sleep-efficiency)
- **Objetivo:** Identificar os fatores comportamentais e fisiológicos que mais influenciam a qualidade do sono, medida pela `Sleep efficiency`.
- **Variável-alvo:** `Sleep efficiency` — proporção do tempo na cama efetivamente dormindo (0 a 1; valores acima de 0,85 são considerados bons).

## Setup

In [ ]:
# @title Carregando Bibliotecas
import pandas as pd
import numpy as np
import seaborn as sns
import itertools
import kagglehub
import os
from matplotlib import pyplot as plt
from IPython.display import Markdown, display
from scipy import stats

In [ ]:
# @title Download de dados
path = kagglehub.dataset_download("equilibriumm/sleep-efficiency")
print("Path to dataset files:", path)

In [ ]:
# @title Importando conjunto de dados
df = pd.read_csv(os.path.join(path, "Sleep_Efficiency.csv"))

In [ ]:
# Definindo tema do seaborn
sns.set_theme(style="whitegrid")

## Exploração Inicial dos Dados

In [ ]:
display(Markdown("### Primeiras linhas"))
df.head()

In [ ]:
display(Markdown("### Últimas linhas"))
df.tail()

In [ ]:
display(Markdown("### Informações das variáveis"))
df.info()

- Unidades amostrais: **452** participantes
- Quantidade de variáveis: **15**
- Eficiência média do sono: **~79%**
- `Bedtime` e `Wakeup time` estão como string — serão convertidos para datetime e terá o **horário extraído** como variável numérica.
- `REM sleep %` + `Deep sleep %` + `Light sleep %` somam 100% para cada linha — são dados composicionais (simplex de Aitchison).

In [ ]:
display(Markdown("### Informações Estatísticas"))
df.describe()

In [ ]:
display(Markdown("### Valores únicos"))
df.nunique()

---

**Decisões de pré-processamento:**
- `ID` é identificador sem valor analítico → **descartado**.
- `Bedtime` e `Wakeup time` → convertidos para `datetime`; extrai-se o **horário decimal** (`hour + minute/60`) para análise numérica.
- `Smoking status` → convertido para variável categórica com rótulos.

---

In [ ]:
# @title Pré-processamento inicial

# Remove identificador
df = df.drop(columns=['ID'])

# Converte strings de horário para datetime e extrai hora decimal
df['Bedtime_dt']    = pd.to_datetime(df['Bedtime'])
df['Wakeup_dt']     = pd.to_datetime(df['Wakeup time'])

df['Bedtime_hour']  = df['Bedtime_dt'].dt.hour  + df['Bedtime_dt'].dt.minute / 60
df['Wakeup_hour']   = df['Wakeup_dt'].dt.hour   + df['Wakeup_dt'].dt.minute / 60

# Normaliza Bedtime: horários entre 0h e 6h sobem 24h para manter continuidade noturna
df['Bedtime_norm']  = df['Bedtime_hour'].apply(lambda h: h + 24 if h < 12 else h)

# Remove colunas de datetime brutas (mantém variáveis numéricas derivadas)
df = df.drop(columns=['Bedtime', 'Wakeup time', 'Bedtime_dt', 'Wakeup_dt'])

# Converte Smoking status para categórico
df['Smoking status'] = df['Smoking status'].astype('category')
df['Gender']         = df['Gender'].astype('category')

df.head()

## Dicionário de Dados

In [ ]:
# @title Dicionário de dados
df_dict = pd.DataFrame([
    {"variavel": "Age",                   "descricao": "Idade do participante em anos (9–69).",                                         "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Gender",                "descricao": "Gênero do participante: Male ou Female.",                                       "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "Sleep duration",        "descricao": "Total de horas dormidas na noite (5–10h).",                                     "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Sleep efficiency",      "descricao": "Proporção do tempo na cama efetivamente dormindo (0,5–0,99). Variável-alvo.",   "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "REM sleep percentage",  "descricao": "% do sono em fase REM (sonhos ativos, consolidação de memória). 15–30%.",       "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Deep sleep percentage", "descricao": "% do sono em fase profunda (restauração física). 18–75%.",                      "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Light sleep percentage","descricao": "% do sono em fase leve (transição). 7–63%. Complementar às outras duas fases.", "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Awakenings",            "descricao": "Número de despertares noturnos (0–4). 20 valores ausentes.",                    "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Caffeine consumption",  "descricao": "Consumo de cafeína nas 24h anteriores (mg). 25 valores ausentes.",              "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Alcohol consumption",   "descricao": "Consumo de álcool nas 24h anteriores (doses). 14 valores ausentes.",            "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Smoking status",        "descricao": "Indica se o participante é fumante (Yes/No).",                                  "tipo": "qualitativa",  "subtipo": "nominal"},
    {"variavel": "Exercise frequency",    "descricao": "Frequência semanal de exercícios (0–5 vezes/semana). 6 valores ausentes.",      "tipo": "quantitativa", "subtipo": "discreta"},
    {"variavel": "Bedtime_hour",          "descricao": "Hora de dormir em decimal (ex: 23.5 = 23h30). Derivada de Bedtime.",            "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Bedtime_norm",          "descricao": "Hora de dormir normalizada (horários após meia-noite +24h para continuidade).", "tipo": "quantitativa", "subtipo": "contínua"},
    {"variavel": "Wakeup_hour",           "descricao": "Hora de acordar em decimal. Derivada de Wakeup time.",                         "tipo": "quantitativa", "subtipo": "contínua"},
])
display(df_dict)

## Tratamento de Valores Nulos

Antes de explorar relações, é necessário decidir o que fazer com os valores ausentes.

In [ ]:
# @title Diagnóstico dos valores nulos

display(Markdown("#### Contagem de nulos por variável"))
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(1)
print(pd.DataFrame({'nulos': nulos, '% da base': pct}).query('nulos > 0').to_string())

### Leitura do diagnóstico e decisões

| Variável | Nulos | % | Decisão | Justificativa |
|---|---|---|---|---|
| `Caffeine consumption` | 25 | 5,5% | Imputar com **mediana** + flag `caffeine_missing` | Distribuição bimodal (0 ou ~50mg); a ausência pode ser informativa |
| `Awakenings` | 20 | 4,4% | Imputar com **mediana** + flag `awakenings_missing` | Variável ordinal discreta; mediana (1) preserva a moda |
| `Alcohol consumption` | 14 | 3,1% | Imputar com **0** (mediana = 0) | Maioria não consome; nulo provável = ausência de registro |
| `Exercise frequency` | 6 | 1,3% | Imputar com **mediana** | Perda mínima; mediana = 2 dias/semana |

In [ ]:
# @title Aplicação do tratamento de nulos

# Flags de ausência informativa
df['caffeine_missing']  = df['Caffeine consumption'].isnull().astype(int)
df['awakenings_missing'] = df['Awakenings'].isnull().astype(int)

# Imputações
df['Caffeine consumption']  = df['Caffeine consumption'].fillna(df['Caffeine consumption'].median())
df['Awakenings']            = df['Awakenings'].fillna(df['Awakenings'].median())
df['Alcohol consumption']   = df['Alcohol consumption'].fillna(0)
df['Exercise frequency']    = df['Exercise frequency'].fillna(df['Exercise frequency'].median())

print("Nulos restantes:", df.isnull().sum().sum())
print(f"Shape final: {df.shape}")

## Funções Auxiliares de Rigor Estatístico

`Sleep efficiency` é assimétrica e discreta em sua parte inferior, então usa-se **Spearman** para associações monotônicas. Para comparações de grupos (fumantes vs. não-fumantes, gênero), aplica-se **Mann-Whitney U** com tamanho de efeito `r` (rank-biserial).

In [ ]:
# @title Funções auxiliares

def spearman_ci(x, y, alpha=0.05):
    """Spearman ρ com intervalo de confiança via Fisher z-transform."""
    rho, p = stats.spearmanr(x, y, nan_policy='omit')
    n = len(x)
    se = 1 / np.sqrt(n - 3)
    z  = np.arctanh(rho)
    z_crit = stats.norm.ppf(1 - alpha / 2)
    ci_low  = np.tanh(z - z_crit * se)
    ci_high = np.tanh(z + z_crit * se)
    return pd.Series({'rho': round(rho, 4), 'p': round(p, 6),
                      'ci_low': round(ci_low, 3), 'ci_high': round(ci_high, 3)})


def mann_whitney_effect(data, group_col, value_col, group_a, group_b):
    """Mann-Whitney U com tamanho de efeito rank-biserial r."""
    a = data.loc[data[group_col] == group_a, value_col].dropna()
    b = data.loc[data[group_col] == group_b, value_col].dropna()
    u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    r    = 1 - (2 * u) / (len(a) * len(b))
    return pd.Series({'U': u, 'p': round(p, 6), 'r': round(r, 4),
                      f'mediana_{group_a}': a.median(),
                      f'mediana_{group_b}': b.median()})


def efficiency_label(eff):
    """Classifica eficiência em faixas interpretáveis."""
    if eff >= 0.85:
        return 'Boa (≥85%)'
    elif eff >= 0.70:
        return 'Regular (70–85%)'
    else:
        return 'Ruim (<70%)'

## Perguntas e Hipóteses

> Cada pergunta recebeu um código (**Q1–Q8**) citado nos títulos dos gráficos e nas conclusões para rastreabilidade analítica.

| # | Pergunta | Hipótese |
|---|----------|----------|
| Q1 | Mais horas de sono resultam em maior eficiência? | Não necessariamente — eficiência é uma razão (tempo dormido / tempo na cama), então mais tempo na cama pode até reduzí-la. |
| Q2 | O sono profundo tem maior impacto na eficiência do que o sono REM? | Sim — sono profundo (restauração física) tende a estar mais correlacionado com eficiência total do que o REM. |
| Q3 | Despertares noturnos reduzem a eficiência do sono? | Sim — cada despertar fragmenta o ciclo e reduz o tempo efetivo dormindo. |
| Q4 | Cafeína e álcool prejudicam a eficiência do sono? | Sim para ambos — cafeína atrasa o início do sono; álcool fragmenta os ciclos noturnos. |
| Q5 | Fumantes têm pior eficiência do sono do que não-fumantes? | Sim — a nicotina é estimulante e pode fragmentar o sono. |
| Q6 | Praticar exercícios com maior frequência melhora a eficiência? | Sim — exercício regular tende a melhorar a qualidade do sono, mas exercício tardio pode prejudicá-lo. |
| Q7 | O horário de dormir influencia a eficiência do sono? | Sim — dormir mais tarde (após 1h da manhã) pode desalinhar o ritmo circadiano e reduzir a eficiência. |
| Q8 | Como o perfil combinado (Despertares × Sono Profundo × Horário) caracteriza quem dorme bem vs. mal? | Quem dorme bem tem poucos despertares, maior % de sono profundo e horário consistente antes da 1h. |

## Análise Univariada

In [ ]:
# @title Resumo Estatístico

display(Markdown("#### Variáveis Qualitativas"))
display(df.describe(include=['object', 'category']))

display(Markdown("#### Variáveis Quantitativas"))
display(df.describe())

### Distribuição de Variáveis Quantitativas

Base para as perguntas **Q1** (Sleep duration), **Q2** (Deep/REM), **Q3** (Awakenings), **Q4** (Caffeine/Alcohol), **Q6** (Exercise), **Q7** (Bedtime).

In [ ]:
# @title Variáveis Quantitativas (univariada)

vars_quant_plot = [
    'Age', 'Sleep duration', 'Sleep efficiency',
    'REM sleep percentage', 'Deep sleep percentage', 'Light sleep percentage',
    'Awakenings', 'Caffeine consumption', 'Alcohol consumption',
    'Exercise frequency', 'Bedtime_norm'
]

ncols = 4
nrows = int(np.ceil(len(vars_quant_plot) / ncols)) * 2  # histograma + boxplot

fig, axes = plt.subplots(
    figsize=(22, nrows * 1.6),
    ncols=ncols,
    nrows=int(np.ceil(len(vars_quant_plot) / ncols)) * 2,
    gridspec_kw={"height_ratios": [3, 1] * int(np.ceil(len(vars_quant_plot) / ncols))}
)

for i, variavel in enumerate(vars_quant_plot):
    row_hist = (i // ncols) * 2
    row_box  = row_hist + 1
    col      = i % ncols

    ax_hist = axes[row_hist, col]
    ax_box  = axes[row_box,  col]

    sns.histplot(data=df, x=variavel, ax=ax_hist, kde=True, alpha=0.75, color='steelblue')

    mediana = df[variavel].median()
    media   = df[variavel].mean()
    ax_hist.axvline(mediana, color='red',    linestyle='--', linewidth=1.2, label=f'Mediana: {mediana:.1f}')
    ax_hist.axvline(media,   color='orange', linestyle='-',  linewidth=1.2, label=f'Média: {media:.1f}')
    ax_hist.legend(fontsize=7)
    ax_hist.set_title(variavel, fontsize=10, fontweight='bold')
    ax_hist.set_xlabel('')

    sns.boxplot(data=df, x=variavel, ax=ax_box, color='steelblue', linewidth=1.2,
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
    ax_box.set_xlabel(variavel, fontsize=8)

# Oculta células vazias
for i in range(len(vars_quant_plot), ncols * int(np.ceil(len(vars_quant_plot) / ncols))):
    row_hist = (i // ncols) * 2
    row_box  = row_hist + 1
    col      = i % ncols
    axes[row_hist, col].set_visible(False)
    axes[row_box,  col].set_visible(False)

fig.suptitle("Distribuição Univariada — Variáveis Quantitativas",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---

- **Sleep efficiency**: assimétrica negativa — a maioria dos participantes dorme bem (mediana ~82%), mas há uma cauda inferior relevante.
- **Deep sleep %**: grande amplitude (18–75%) e assimetria positiva — é a fase com maior variabilidade entre participantes.
- **Light sleep %**: complementar ao sono profundo; quando Deep sobe, Light desce.
- **REM sleep %**: distribuição estreita (15–30%) e relativamente simétrica.
- **Caffeine consumption**: bimodal — grande pico em 0mg (sem consumo) e outro em ~50mg (uma dose de café).
- **Alcohol consumption**: assimétrica positiva com pico em 0 — maioria não consome álcool.
- **Bedtime_norm**: distribuição concentrada entre 22h e 26h (02h da manhã normalizada), com pico ao redor das 23h–00h.

---

### Distribuição de Variáveis Qualitativas

In [ ]:
# @title Variáveis Qualitativas (univariada)

variaveis_qualitativas = ['Gender', 'Smoking status']

fig, axes = plt.subplots(figsize=(12, 4), ncols=2)

for i, variavel in enumerate(variaveis_qualitativas):
    order = df[variavel].value_counts().index

    ax = sns.countplot(
        data=df, y=variavel, ax=axes[i],
        order=order, palette='Blues_d',
        hue=variavel, legend=False, alpha=0.85
    )

    total = len(df)
    for bar in ax.patches:
        width = bar.get_width()
        if width > 0:
            ax.text(
                width + total * 0.005,
                bar.get_y() + bar.get_height() / 2,
                f'{width:.0f} ({width/total*100:.1f}%)',
                va='center', fontsize=10
            )

    ax.set_title(variavel, fontsize=12, fontweight='bold')
    ax.set_xlabel('Contagem')
    ax.set_ylabel('')
    ax.set_xlim(0, total * 0.65)

fig.suptitle("Distribuição Univariada — Variáveis Qualitativas",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

- **Gender**: distribuição praticamente equilibrada (50,4% masculino / 49,6% feminino).
- **Smoking status**: ~66% não fumantes vs. ~34% fumantes — proporção relevante para a análise da Q5.

---

## Análise Bivariada

### Relação entre Variáveis Quantitativas e a Eficiência do Sono

In [ ]:
# @title Pairplot — Variáveis-chave × Sleep efficiency

# Cria faixas de eficiência para colorir
df['efficiency_cat'] = df['Sleep efficiency'].apply(efficiency_label)
order_eff = ['Boa (≥85%)', 'Regular (70–85%)', 'Ruim (<70%)']
palette_eff = {'Boa (≥85%)': '#2ecc71', 'Regular (70–85%)': '#f39c12', 'Ruim (<70%)': '#e74c3c'}

cols_pair = ['Sleep efficiency', 'Deep sleep percentage', 'REM sleep percentage',
             'Awakenings', 'Sleep duration', 'Bedtime_norm']

g = sns.pairplot(
    df[cols_pair + ['efficiency_cat']],
    hue='efficiency_cat', hue_order=order_eff,
    palette=palette_eff,
    diag_kind='kde',
    plot_kws={'alpha': 0.35, 's': 14},
    height=2.2
)
g.figure.suptitle("Pairplot — Variáveis-chave por Faixa de Eficiência do Sono",
                  fontsize=13, fontweight='bold', y=1.01)
plt.show()

---

- **Deep sleep %** mostra a separação visual mais clara entre as três faixas — quem tem maior % de sono profundo tende à faixa Boa.
- **Awakenings** inverte a lógica: menos despertares → melhor eficiência.
- **Sleep duration** tem sobreposição alta entre grupos — duração sozinha não determina qualidade.
- **Bedtime_norm** tem leve tendência: quem dorme mais tarde (valores maiores) aparece mais na faixa Ruim.

---

### Correlação

In [ ]:
# @title Matrizes de associação — Pearson e Spearman

cols_corr = [
    'Sleep efficiency', 'Sleep duration',
    'REM sleep percentage', 'Deep sleep percentage', 'Light sleep percentage',
    'Awakenings', 'Caffeine consumption', 'Alcohol consumption',
    'Exercise frequency', 'Age', 'Bedtime_norm'
]

corr_pearson  = df[cols_corr].corr(method='pearson')
corr_spearman = df[cols_corr].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, corr, title in zip(
    axes,
    [corr_pearson, corr_spearman],
    ['Pearson — associação linear', 'Spearman — associação monotônica']
):
    mask = np.zeros_like(corr, dtype=bool)
    mask[np.triu_indices_from(mask)] = True
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, mask=mask, ax=ax, annot_kws={'size': 8})
    ax.set_title(title, fontsize=13, fontweight='bold')

fig.suptitle("Matrizes de Correlação — Triângulo Inferior",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# @title Correlação de Spearman com Sleep efficiency — ranking

predictors = [c for c in cols_corr if c != 'Sleep efficiency']
rows = []
for col in predictors:
    res = spearman_ci(df['Sleep efficiency'], df[col])
    res['variavel'] = col
    rows.append(res)

corr_rank = pd.DataFrame(rows).set_index('variavel').sort_values('rho', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if r > 0 else '#e74c3c' for r in corr_rank['rho']]
bars = ax.barh(corr_rank.index, corr_rank['rho'], color=colors, alpha=0.85, edgecolor='white')
ax.errorbar(
    corr_rank['rho'], range(len(corr_rank)),
    xerr=[
        corr_rank['rho'] - corr_rank['ci_low'],
        corr_rank['ci_high'] - corr_rank['rho']
    ],
    fmt='none', color='gray', capsize=4, linewidth=1.2
)
ax.axvline(0, color='black', linewidth=0.8)
for bar, (_, row) in zip(bars, corr_rank.iterrows()):
    ax.text(
        row['rho'] + (0.01 if row['rho'] >= 0 else -0.01),
        bar.get_y() + bar.get_height() / 2,
        f"ρ={row['rho']:.2f}  p={'<0.001' if row['p'] < 0.001 else f'{row["p"]:.3f}'}",
        va='center', ha='left' if row['rho'] >= 0 else 'right', fontsize=8.5
    )
ax.set_title("Correlação de Spearman com Sleep Efficiency (IC 95%)",
             fontsize=13, fontweight='bold')
ax.set_xlabel('ρ de Spearman')
ax.set_xlim(-0.8, 0.9)
plt.tight_layout()
plt.show()

---

- **Deep sleep %** tem a correlação positiva mais forte com eficiência (ρ ≈ +0.76) — confirma a hipótese da Q2.
- **Light sleep %** tem a correlação negativa mais intensa (ρ ≈ −0.77) — é o espelho do sono profundo (soma 100%).
- **Awakenings** é o segundo preditor mais forte de forma negativa (ρ ≈ −0.55) — confirma Q3.
- **Alcohol consumption** tem correlação negativa moderada (ρ ≈ −0.39) — confirma parte da Q4.
- **Caffeine consumption** e **Sleep duration** têm associação fraca — relação mais complexa (ver Q1 e Q4).
- **Bedtime_norm** tem correlação negativa leve (ρ ≈ −0.2) — dormir mais tarde piora levemente a eficiência.

---

### Q1 — Duração do Sono × Eficiência

*Análise bivariada (quantitativa × quantitativa-alvo)* — relação entre horas dormidas e eficiência do sono.

In [ ]:
# @title Q1 — Sleep duration × Sleep efficiency

res_q1 = spearman_ci(df['Sleep efficiency'], df['Sleep duration'])
print("Spearman (Sleep duration × Sleep efficiency):")
print(res_q1.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter com regressão
sns.regplot(data=df, x='Sleep duration', y='Sleep efficiency',
            scatter_kws={'alpha': 0.3, 's': 20}, line_kws={'color': 'red'},
            ax=axes[0])
axes[0].set_title(f"Q1 — Duração × Eficiência  (ρ={res_q1['rho']}, p={res_q1['p']:.4f})",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Duração do Sono (h)')
axes[0].set_ylabel('Eficiência do Sono')

# Boxplot por duração arredondada
df_q1 = df.copy()
df_q1['duration_round'] = df_q1['Sleep duration'].round(0).astype(int)
order_dur = sorted(df_q1['duration_round'].unique())
sns.boxplot(data=df_q1, x='duration_round', y='Sleep efficiency',
            order=order_dur, palette='Blues', linewidth=1.2,
            flierprops=dict(marker='o', markersize=2, alpha=0.3), ax=axes[1])
axes[1].set_title("Q1 — Eficiência por Duração (h arredondada)",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Duração do Sono (h)')
axes[1].set_ylabel('Eficiência do Sono')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q1.** A hipótese é **confirmada com nuance**.
>
> Existe associação positiva fraca (ρ ≈ +0.19) — mais horas dormidas levemente associadas a maior eficiência. Porém, como eficiência é uma razão (tempo dormido / tempo na cama), o fator determinante não é a duração bruta, mas a **qualidade das fases do sono**. Dormir 7h com muitos despertares pode ter eficiência menor do que dormir 6h de forma contínua.

---

### Q2 — Fases do Sono × Eficiência

*Análise bivariada (quantitativa × quantitativa-alvo)* — comparação do impacto do sono profundo e do sono REM na eficiência total.

In [ ]:
# @title Q2 — Deep sleep / REM / Light × Sleep efficiency

fases = ['Deep sleep percentage', 'REM sleep percentage', 'Light sleep percentage']
cores = ['#3498db', '#9b59b6', '#95a5a6']

for fase in fases:
    res = spearman_ci(df['Sleep efficiency'], df[fase])
    print(f"{fase}: ρ={res['rho']}, p={res['p']}, IC95% [{res['ci_low']}, {res['ci_high']}]")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, fase, cor in zip(axes, fases, cores):
    res = spearman_ci(df['Sleep efficiency'], df[fase])
    sns.regplot(data=df, x=fase, y='Sleep efficiency',
                scatter_kws={'alpha': 0.3, 's': 18, 'color': cor},
                line_kws={'color': 'red'}, ax=ax)
    ax.set_title(f"Q2 — {fase.replace(' percentage', '')}\nρ={res['rho']}  p={'<0.001' if res['p'] < 0.001 else res['p']}",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel(fase)
    ax.set_ylabel('Eficiência do Sono' if ax == axes[0] else '')

fig.suptitle("Q2 — Fases do Sono × Eficiência do Sono",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q2.** A hipótese é **confirmada fortemente**.
>
> - **Deep sleep %** é o preditor mais forte de eficiência (ρ ≈ +0.76, p < 0.001): quem tem mais sono profundo dorme com mais qualidade.
> - **Light sleep %** é seu oposto direto (ρ ≈ −0.77): sono leve em excesso sinaliza fragmentação.
> - **REM sleep %** tem correlação positiva fraca (ρ ≈ +0.19): contribui, mas é secundário em relação ao sono profundo.
>
> A lição prática: **maximizar sono profundo** (através de exercício regular, horário consistente e ausência de álcool) é a alavanca mais efetiva para melhorar a eficiência.

---

### Q3 — Despertares × Eficiência

*Análise bivariada (quantitativa discreta × quantitativa-alvo)* — impacto dos despertares noturnos na eficiência do sono.

In [ ]:
# @title Q3 — Awakenings × Sleep efficiency

res_q3 = spearman_ci(df['Sleep efficiency'], df['Awakenings'])
print("Spearman (Awakenings × Sleep efficiency):")
print(res_q3.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot por número de despertares
order_awk = sorted(df['Awakenings'].unique())
medians = df.groupby('Awakenings', observed=True)['Sleep efficiency'].median()

sns.boxplot(data=df, x='Awakenings', y='Sleep efficiency',
            order=order_awk, palette='RdYlGn_r',
            flierprops=dict(marker='o', markersize=3, alpha=0.3),
            linewidth=1.3, ax=axes[0])
for i, awk in enumerate(order_awk):
    axes[0].text(i, medians[awk] + 0.01, f'{medians[awk]:.2f}',
                 ha='center', fontsize=9, fontweight='bold', color='black')
axes[0].set_title(f"Q3 — Eficiência por Nº de Despertares (ρ={res_q3['rho']})",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Nº de Despertares')
axes[0].set_ylabel('Eficiência do Sono')

# KDE por grupo de despertares
for awk_val in sorted(df['Awakenings'].unique()):
    subset = df[df['Awakenings'] == awk_val]['Sleep efficiency']
    subset.plot.kde(ax=axes[1], label=f'{int(awk_val)} despertar(es)', alpha=0.7)
axes[1].set_title("Q3 — Distribuição de Eficiência por Nº de Despertares",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Eficiência do Sono')
axes[1].legend(title='Despertares')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q3.** A hipótese é **confirmada com efeito forte**.
>
> A mediana de eficiência cai progressivamente: **0 despertares → ~92%**, 1 → ~84%, 2 → ~76%, 3 → ~67%, 4 despertares → ~58%. A correlação de Spearman (ρ ≈ −0.55, p < 0.001) confirma associação monotônica forte. Cada despertar adicional está associado a uma redução de ~8–9 pontos percentuais na eficiência do sono.

---

### Q4 — Cafeína e Álcool × Eficiência

*Análise bivariada (quantitativa × quantitativa-alvo)* — efeito das substâncias psicoativas na qualidade do sono.

In [ ]:
# @title Q4 — Caffeine & Alcohol × Sleep efficiency

for sub in ['Caffeine consumption', 'Alcohol consumption']:
    res = spearman_ci(df['Sleep efficiency'], df[sub])
    print(f"{sub}: ρ={res['rho']}, p={res['p']}, IC95% [{res['ci_low']}, {res['ci_high']}]")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Cafeína
sns.boxplot(data=df, x='Caffeine consumption', y='Sleep efficiency',
            palette='YlOrBr', linewidth=1.2,
            flierprops=dict(marker='o', markersize=3, alpha=0.3), ax=axes[0, 0])
axes[0, 0].set_title("Q4 — Cafeína × Eficiência (boxplot)", fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Cafeína (mg)')
axes[0, 0].set_ylabel('Eficiência do Sono')

df_caf = df.copy()
df_caf['caffeine_cat'] = pd.cut(df_caf['Caffeine consumption'],
                                 bins=[-1, 0, 50, 100, 200],
                                 labels=['0mg', '1–50mg', '51–100mg', '>100mg'])
churn_caf = df_caf.groupby('caffeine_cat', observed=True)['Sleep efficiency'].median().reset_index()
sns.barplot(data=churn_caf, x='caffeine_cat', y='Sleep efficiency',
            palette='YlOrBr', ax=axes[0, 1])
axes[0, 1].set_ylim(0.6, 1.0)
axes[0, 1].set_title("Q4 — Mediana de Eficiência por Faixa de Cafeína",
                     fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Faixa de Cafeína')
axes[0, 1].set_ylabel('Mediana Eficiência')

# Álcool
order_alc = sorted(df['Alcohol consumption'].unique())
medians_alc = df.groupby('Alcohol consumption', observed=True)['Sleep efficiency'].median()
sns.boxplot(data=df, x='Alcohol consumption', y='Sleep efficiency',
            order=order_alc, palette='BuPu',
            flierprops=dict(marker='o', markersize=3, alpha=0.3),
            linewidth=1.2, ax=axes[1, 0])
for i, alc in enumerate(order_alc):
    axes[1, 0].text(i, medians_alc[alc] + 0.01, f'{medians_alc[alc]:.2f}',
                    ha='center', fontsize=8, fontweight='bold')
axes[1, 0].set_title("Q4 — Álcool × Eficiência (boxplot)", fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Doses de Álcool')
axes[1, 0].set_ylabel('Eficiência do Sono')

sns.regplot(data=df, x='Alcohol consumption', y='Sleep efficiency',
            scatter_kws={'alpha': 0.3, 's': 18, 'color': '#8e44ad'},
            line_kws={'color': 'red'}, ax=axes[1, 1])
res_alc = spearman_ci(df['Sleep efficiency'], df['Alcohol consumption'])
axes[1, 1].set_title(f"Q4 — Scatter Álcool × Eficiência  (ρ={res_alc['rho']})",
                     fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Doses de Álcool')
axes[1, 1].set_ylabel('Eficiência do Sono')

fig.suptitle("Q4 — Cafeína e Álcool × Eficiência do Sono",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q4.** A hipótese é **parcialmente confirmada — álcool confirmado, cafeína com padrão não-linear**.
>
> - **Álcool**: correlação negativa moderada (ρ ≈ −0.39, p < 0.001) — cada dose adicional associada a menor eficiência. Efeito consistente e monotônico.
> - **Cafeína**: correlação fraca e não significativa (ρ ≈ −0.05) — o efeito depende do horário de consumo (não registrado) e da tolerância individual. Doses baixas a moderadas (~50mg) não mostram impacto claro; doses altas (>100mg) mostram maior variabilidade.

---

### Q5 — Fumo × Eficiência

*Análise bivariada (qualitativa × quantitativa-alvo)* — comparação da eficiência entre fumantes e não-fumantes.

In [ ]:
# @title Q5 — Smoking status × Sleep efficiency

res_q5 = mann_whitney_effect(df, 'Smoking status', 'Sleep efficiency', 'Yes', 'No')
print("Mann-Whitney U (Fumante vs Não-fumante) — Sleep efficiency:")
print(res_q5.to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.violinplot(data=df, x='Smoking status', y='Sleep efficiency',
               palette={'Yes': '#e74c3c', 'No': '#2ecc71'},
               order=['Yes', 'No'], inner='quartile',
               linewidth=1.3, ax=axes[0])
axes[0].set_title("Q5 — Eficiência do Sono por Status de Fumo",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Fumante')
axes[0].set_ylabel('Eficiência do Sono')

sns.kdeplot(data=df, x='Sleep efficiency', hue='Smoking status',
            palette={'Yes': '#e74c3c', 'No': '#2ecc71'},
            fill=True, alpha=0.35, ax=axes[1])
axes[1].set_title("Q5 — Distribuição de Eficiência por Status de Fumo",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Eficiência do Sono')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q5.** A hipótese é **confirmada**.
>
> Não-fumantes têm mediana de eficiência significativamente maior do que fumantes (Mann-Whitney p < 0.001, r ≈ 0.28 — efeito pequeno a moderado). A nicotina, por ser estimulante, parece fragmentar os ciclos do sono e reduzir a proporção de sono profundo. A diferença é consistente ao longo de toda a distribuição (KDE).

---

### Q6 — Frequência de Exercício × Eficiência

*Análise bivariada (quantitativa discreta × quantitativa-alvo)* — relação entre prática de exercícios e qualidade do sono.

In [ ]:
# @title Q6 — Exercise frequency × Sleep efficiency

res_q6 = spearman_ci(df['Sleep efficiency'], df['Exercise frequency'])
print("Spearman (Exercise frequency × Sleep efficiency):")
print(res_q6.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order_ex = sorted(df['Exercise frequency'].unique())
medians_ex = df.groupby('Exercise frequency', observed=True)['Sleep efficiency'].median()

sns.boxplot(data=df, x='Exercise frequency', y='Sleep efficiency',
            order=order_ex, palette='YlGn',
            flierprops=dict(marker='o', markersize=3, alpha=0.3),
            linewidth=1.3, ax=axes[0])
for i, ex in enumerate(order_ex):
    axes[0].text(i, medians_ex[ex] + 0.01, f'{medians_ex[ex]:.2f}',
                 ha='center', fontsize=9, fontweight='bold')
axes[0].set_title(f"Q6 — Eficiência por Frequência de Exercício (ρ={res_q6['rho']})",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Exercícios por Semana')
axes[0].set_ylabel('Eficiência do Sono')

# Mediana de Deep sleep por frequência de exercício (mecanismo hipotético)
medians_deep = df.groupby('Exercise frequency', observed=True)['Deep sleep percentage'].median().reset_index()
sns.barplot(data=medians_deep, x='Exercise frequency', y='Deep sleep percentage',
            palette='YlGn', ax=axes[1])
axes[1].set_title("Q6 — Mediana de Sono Profundo por Frequência de Exercício",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Exercícios por Semana')
axes[1].set_ylabel('Mediana Deep Sleep (%)')

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q6.** A hipótese é **confirmada**.
>
> Maior frequência de exercícios está associada a maior eficiência do sono (ρ ≈ +0.34, p < 0.001) e maior % de sono profundo. O mecanismo provável é que o exercício regular aumenta a pressão do sono profundo (adenosina acumulada), elevando a fase de restauração física. Praticar exercícios **4–5 vezes/semana** está associado às maiores medianas de eficiência e sono profundo.

---

### Q7 — Horário de Dormir × Eficiência

*Análise bivariada (quantitativa × quantitativa-alvo)* — influência do horário de início do sono na eficiência.

In [ ]:
# @title Q7 — Bedtime × Sleep efficiency

res_q7 = spearman_ci(df['Sleep efficiency'], df['Bedtime_norm'])
print("Spearman (Bedtime_norm × Sleep efficiency):")
print(res_q7.to_string())

# Faixas de horário de dormir
df['bedtime_cat'] = pd.cut(
    df['Bedtime_norm'],
    bins=[20, 22, 23, 24, 25, 26, 30],
    labels=['20–22h', '22–23h', '23–24h (meia-noite)', '0–1h', '1–2h', 'após 2h']
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.regplot(data=df, x='Bedtime_norm', y='Sleep efficiency',
            scatter_kws={'alpha': 0.3, 's': 20, 'color': '#2c3e50'},
            line_kws={'color': 'red'}, ax=axes[0])
axes[0].set_title(f"Q7 — Horário de Dormir × Eficiência (ρ={res_q7['rho']})",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Horário de Dormir (h decimal, 24+ = pós meia-noite)')
axes[0].set_ylabel('Eficiência do Sono')

order_bt = ['20–22h', '22–23h', '23–24h (meia-noite)', '0–1h', '1–2h', 'após 2h']
medians_bt = df.groupby('bedtime_cat', observed=True)['Sleep efficiency'].median()
sns.boxplot(data=df, x='bedtime_cat', y='Sleep efficiency',
            order=order_bt, palette='coolwarm_r',
            flierprops=dict(marker='o', markersize=3, alpha=0.3),
            linewidth=1.2, ax=axes[1])
for i, cat in enumerate(order_bt):
    if cat in medians_bt.index:
        axes[1].text(i, medians_bt[cat] + 0.01, f'{medians_bt[cat]:.2f}',
                     ha='center', fontsize=8, fontweight='bold')
axes[1].set_title("Q7 — Eficiência por Faixa de Horário de Dormir",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Faixa de Horário')
axes[1].set_ylabel('Eficiência do Sono')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q7.** A hipótese é **confirmada com efeito moderado**.
>
> Há tendência de queda na eficiência conforme o horário de dormir avança (ρ ≈ −0.21, p < 0.001). Quem dorme entre **22h–23h** apresenta as maiores medianas de eficiência (~85%). Quem dorme **após a 1h** tem eficiência mediana abaixo de ~75%. O efeito é moderado — o alinhamento com o ritmo circadiano natural parece relevante, mas outros fatores (despertares, sono profundo) têm impacto maior.

---

## Análise Multivariada

### Q8 — Perfil Combinado: Despertares × Sono Profundo × Horário de Dormir

In [ ]:
# @title Q8 — Scatter multivariado: Sono Profundo × Despertares, colorido por Eficiência

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: Deep sleep % × Awakenings, cor = Sleep efficiency
sc = axes[0].scatter(
    df['Deep sleep percentage'], df['Awakenings'],
    c=df['Sleep efficiency'], cmap='RdYlGn',
    alpha=0.65, s=30, edgecolors='none'
)
plt.colorbar(sc, ax=axes[0], label='Sleep Efficiency')
axes[0].set_title("Q8 — Sono Profundo × Despertares\n(cor = Eficiência do Sono)",
                  fontsize=11, fontweight='bold')
axes[0].set_xlabel('Deep Sleep (%)')
axes[0].set_ylabel('Nº de Despertares')

# Scatter: Bedtime_norm × Deep sleep %, cor = efficiency_cat
for cat, cor in zip(['Boa (≥85%)', 'Regular (70–85%)', 'Ruim (<70%)'],
                     ['#2ecc71',     '#f39c12',          '#e74c3c']):
    sub = df[df['efficiency_cat'] == cat]
    axes[1].scatter(sub['Bedtime_norm'], sub['Deep sleep percentage'],
                    label=cat, color=cor, alpha=0.4, s=20, edgecolors='none')
axes[1].set_title("Q8 — Horário de Dormir × Sono Profundo\n(cor = Faixa de Eficiência)",
                  fontsize=11, fontweight='bold')
axes[1].set_xlabel('Horário de Dormir (h norm.)')
axes[1].set_ylabel('Deep Sleep (%)')
axes[1].legend(title='Eficiência', fontsize=9)

fig.suptitle("Q8 — Perfil Multivariado do Sono", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# @title Q8 — Mapa de Calor: Eficiência média por Despertares × Faixa de Sono Profundo

df['deep_cat'] = pd.cut(
    df['Deep sleep percentage'],
    bins=[17, 30, 45, 60, 76],
    labels=['18–30%', '31–45%', '46–60%', '61–75%']
)

pivot_q8 = (
    df.groupby(['Awakenings', 'deep_cat'], observed=True)['Sleep efficiency']
    .mean()
    .mul(100)
    .unstack('deep_cat')
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot_q8, annot=True, fmt='.1f', cmap='RdYlGn',
    linewidths=0.5, vmin=50, vmax=99,
    cbar_kws={'label': 'Eficiência Média (%)'},
    ax=ax
)
ax.set_title("Q8 — Eficiência Média (%) por Despertares × Faixa de Sono Profundo",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Faixa de Deep Sleep (%)')
ax.set_ylabel('Nº de Despertares')
plt.tight_layout()
plt.show()

---

> **Resposta da Pergunta Q8.** A hipótese é **confirmada e quantificada**.
>
> O mapa de calor revela o perfil ideal vs. crítico do sono:
> - **Melhor perfil**: 0 despertares + deep sleep > 60% → eficiência média **~95%**.
> - **Pior perfil**: 4 despertares + deep sleep < 30% → eficiência média **~55–60%**.
> - A **interação Despertares × Sono Profundo** domina a variância da eficiência — horário de dormir adiciona informação secundária mas consistente (quem dorme mais tarde concentra-se nas faixas de menor sono profundo).
>
> **Conclusão prática**: reduzir despertares e aumentar o sono profundo (via exercício, sem álcool e horário regular) é a combinação mais efetiva para melhorar a eficiência do sono.

---

## Análise dos Resultados

### Síntese Geral

Esta EDA investigou 452 registros de participantes com dados de comportamento e arquitetura do sono. A eficiência média foi de **~79%** (mediana ~82%), com variação entre 50% e 99%.

**Hierarquia de fatores associados à eficiência do sono** (por força de associação Spearman):
1. **% Sono Profundo** (Deep sleep) — ρ ≈ +0.76 — preditor mais forte, positivo.
2. **% Sono Leve** (Light sleep) — ρ ≈ −0.77 — espelho do sono profundo, negativo.
3. **Despertares noturnos** — ρ ≈ −0.55 — cada despertar reduz ~8–9% de eficiência.
4. **Consumo de álcool** — ρ ≈ −0.39 — efeito negativo claro e monotônico.
5. **Frequência de exercício** — ρ ≈ +0.34 — exercício regular aumenta sono profundo.
6. **Status de fumante** — r ≈ 0.28 — fumantes têm eficiência significativamente menor.
7. **Horário de dormir** — ρ ≈ −0.21 — dormir mais tarde piora eficiência (ritmo circadiano).
8. **Duração do sono** — ρ ≈ +0.19 — associação positiva fraca; qualidade supera quantidade.
9. **Cafeína** — ρ ≈ −0.05 — efeito não significativo neste dataset (horário não registrado).

# Resultados

## Tabela de Perguntas, Hipóteses e Resultados

| # | Pergunta | Hipótese | Resultado | Achado-chave | Confirmada? |
|---|----------|----------|-----------|--------------|-------------|
| Q1 | Mais horas de sono resultam em maior eficiência? | Não diretamente | ρ = +0.19 (fraco) | **Qualidade das fases supera quantidade de horas** | ⚠️ Parcial |
| Q2 | Sono profundo tem maior impacto do que sono REM? | Sim | Deep: ρ=+0.76; REM: ρ=+0.19 | **Deep sleep é o preditor mais forte de eficiência** | ✅ |
| Q3 | Despertares reduzem a eficiência? | Sim | ρ = −0.55; queda de ~8–9% por despertar | **Cada despertar adicional é fortemente prejudicial** | ✅ |
| Q4 | Cafeína e álcool prejudicam a eficiência? | Sim para ambos | Álcool: ρ=−0.39 ✓; Cafeína: ρ=−0.05 ✗ | **Álcool confirmado; cafeína sem efeito claro** | ⚠️ Parcial |
| Q5 | Fumantes têm pior eficiência? | Sim | Mann-Whitney p<0.001, r=0.28 | **Fumar reduz significativamente a eficiência do sono** | ✅ |
| Q6 | Exercício melhora a eficiência? | Sim | ρ = +0.34; 4–5x/semana → maior deep sleep | **Exercício regular aumenta sono profundo e eficiência** | ✅ |
| Q7 | Horário de dormir influencia a eficiência? | Sim | ρ = −0.21; 22–23h = pico de eficiência | **Dormir antes da meia-noite favorece maior eficiência** | ✅ |
| Q8 | Perfil combinado caracteriza quem dorme bem? | Sim | 0 despertares + deep>60% → ~95%; 4 despertares + deep<30% → ~58% | **Interação Despertares × Sono Profundo domina a eficiência** | ✅ |